# 03 – Processed Data Preview (Milestone 2)

Xem trực tiếp artifact cuối cùng của Milestone 2: `data/processed/air_pollution_final.parquet`.

**Vai trò của notebook này — chỉ ĐỌC, không tính toán lại kết quả:**

| Quy tắc | Cụ thể |
|---|---|
| Không tạo cleaning logic mới | Mọi ngưỡng kiểm tra đều **import từ** `src/cleaning.py` và `src/data_collection.py`, không viết literal |
| Không sửa logic cleaning/preprocessing | Không gọi hàm nào của `src/cleaning.py` / `src/cleaning_pipeline.py` ở đây |
| Không sửa file trên đĩa | Không `to_parquet`, không `to_csv` |
| Schema là hợp đồng, không phải gợi ý | `timestamp` / `station_id` **assert** theo `docs/data_dictionary.md`, không dò tên cột |

**Vì sao NaN sau cleaning không phải lỗi.** `src/cleaning.py` xoá quan sát (chuyển thành `NaN`) khi
dữ liệu không đáng tin — cảm biến kẹt, trạm ngừng phát, mã lỗi `-999/-9999`, nghịch đảo khí động học
PM2.5 ⊄ PM10. Vì vậy `NaN` ở đây là **bằng chứng cleaning đã chạy**, không phải hụt dữ liệu.
Ba cột cờ `pm25_was_stuck` / `pm25_was_missing` / `is_high_humidity_fog` là **cờ chẩn đoán, không
phải imputation** — xem `AGENTS.md` §"Diagnostic flags are not imputations".

**Vì sao CI không chạy notebook này.** Artifact `data/processed/*.parquet` bị `.gitignore` loại bỏ
và chỉ sinh ra sau khi chạy `01_data_collection.ipynb` (cần network). Job `notebook-smoke` trong
`.github/workflows/ci.yml` chỉ execute `00_environment_test.ipynb`. Muốn xác minh tự động thì
phải chạy ở local, sau khi 01 → 02 → 03_transformation_pipeline đã hoàn tất.

## 0. Setup & kiểm tra môi trường

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# sys.path phải trỏ tới GỐC REPO để `import src.*` tôn trọng cấu trúc package —
# trỏ vào `src/` rồi import `cleaning` sẽ không hoạt động.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

# Ngưỡng và tên cột lấy từ module thật sự áp dụng chúng. Nếu một module đổi ngưỡng,
# notebook này đổi theo — không bao giờ trôi khỏi nhau.
from src.cleaning import (  # noqa: E402
    DISGUISED_NUMERIC_CODES,
    HIGH_HUMIDITY_FLAG,
    HIGH_HUMIDITY_FOG_THRESHOLD_PCT,
    PM25_MISSING_FLAG,
    PM_AERODYNAMIC_EPSILON_UG_M3,
    STATION_COLUMN,
    STUCK_SENSOR_FLAG,
    TIMESTAMP_COLUMN,
)
from src.cleaning_pipeline import (  # noqa: E402
    CYCLICAL_FEATURES,
    DIAGNOSTIC_FEATURES,
    NUMERIC_FEATURES,
    TARGET_CANDIDATE,
    TARGET_DERIVED_FLAGS,
)
from src.data_collection import WEATHER_PHYSICAL_BOUNDS  # noqa: E402

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10
sns.set_theme(style="whitegrid")

print("✓ Environment setup complete")
print(f"  REPO_ROOT   : {REPO_ROOT}")
print(f"  ε khí động học: {PM_AERODYNAMIC_EPSILON_UG_M3} µg/m³")
print(f"  Ngưỡng sương mù: RH > {HIGH_HUMIDITY_FOG_THRESHOLD_PCT}%")
print(f"  Mã lỗi      : {DISGUISED_NUMERIC_CODES}")

## 1. Load processed dataset

In [ ]:
OUTPUT_PATH = REPO_ROOT / "data" / "processed" / "air_pollution_final.parquet"

if not OUTPUT_PATH.exists():
    print("❌ FILE KHÔNG TỒN TẠI!")
    print(f"   Đường dẫn: {OUTPUT_PATH}")
    print()
    print("Artifact này bị .gitignore loại bỏ nên phải sinh ra ở local. Thứ tự chạy:")
    print("  1. python scripts/fetch_dataset.py           # hoặc chạy 01_data_collection.ipynb")
    print("  2. 02_quality_audit.ipynb                     # Issue #5 — audit 6 chiều")
    print("  3. 03_data_cleaning.ipynb                    # Issue #6 — làm sạch tất định")
    print("  4. 03_transformation_pipeline.ipynb          # Issue #7 — merge → freeze → split → fit")
    print()
    print("Bước 4 là bước duy nhất ghi ra file parquet này.")
    raise FileNotFoundError(f"File not found: {OUTPUT_PATH}")

df = pd.read_parquet(OUTPUT_PATH)

print(f"✓ Đọc thành công: {OUTPUT_PATH.relative_to(REPO_ROOT)}")
print(f"  Kích thước file: {OUTPUT_PATH.stat().st_size:,} bytes")
print(f"  rows × cols  : {df.shape[0]:,} × {df.shape[1]}")
print(f"  dtype index  : {df.index.dtype}")
print()
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:<24} {df[col].dtype}")

### 1a. Assert schema canonical

`docs/data_dictionary.md` §3.3 định nghĩa **Observation Key = (`station_id`, `timestamp`)** cho tập
quan sát đa trạm. Ở đây ta **không dò tên cột**: tên canonical là hợp đồng, đổi tên là vi phạm
schema và phải nổ lỗi chứ không âm thầm dò sang cột khác.

In [ ]:
missing_schema = [c for c in (TIMESTAMP_COLUMN, STATION_COLUMN) if c not in df.columns]
if missing_schema:
    raise AssertionError(
        f"Thiếu cột canonical {missing_schema}. Dataset không khớp docs/data_dictionary.md §3. "
        f"Cột hiện có: {df.columns.tolist()}"
    )

# Cột thời gian phải là datetime thật, không phải chuỗi — nếu không thì mọi phép
# so sánh mốc cắt ở dưới sẽ so chuỗi với chuỗi và im lặng cho kết quả sai.
assert pd.api.types.is_datetime64_any_dtype(df[TIMESTAMP_COLUMN]), (
    f"{TIMESTAMP_COLUMN} phải là datetime, đang là {df[TIMESTAMP_COLUMN].dtype}"
)
print(f"✓ Schema canonical OK: {TIMESTAMP_COLUMN} ({df[TIMESTAMP_COLUMN].dtype}), "
      f"{STATION_COLUMN} ({df[STATION_COLUMN].dtype})")

# Liệt kê feature nào của pipeline KHÔNG có trong artifact — đây là tín hiệu
# lỗi, phải nêu tên chứ không lặng lẽ lọc bỏ.
expected_features = NUMERIC_FEATURES + CYCLICAL_FEATURES + DIAGNOSTIC_FEATURES + TARGET_DERIVED_FLAGS
absent = [c for c in expected_features if c not in df.columns]
if absent:
    print(f"⚠️  Thiếu {len(absent)} feature mà pipeline kỳ vọng: {absent}")
else:
    print(f"✓ Đủ {len(expected_features)} feature pipeline kỳ vọng")

### 1b. Head / tail / sample

In [ ]:
print("=== HEAD (10 dòng đầu) ===")
df.head(10)

In [ ]:
print("=== TAIL (10 dòng cuối) ===")
df.tail(10)

In [ ]:
print("=== SAMPLE (10 dòng ngẫu nhiên, random_state=42) ===")
df.sample(10, random_state=RANDOM_STATE)

## 2. Dataset schema (chi tiết hơn `df.info()`)

In [ ]:
schema_df = pd.DataFrame([
    {
        "column": col,
        "dtype": str(df[col].dtype),
        "non_null": int(df[col].notna().sum()),
        "missing": int(df[col].isna().sum()),
        "missing_%": round(100 * df[col].isna().mean(), 3),
        "n_unique": int(df[col].nunique(dropna=True)),
    }
    for col in df.columns
])
print("=== DATASET SCHEMA ===")
display(schema_df)

## 3. Thống kê dữ liệu

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"=== THỐNG KÊ {len(numeric_cols)} BIẾN SỐ ===")
display(df[numeric_cols].describe().T)

## 4. Kiểm tra thời gian

In [ ]:
ts = df[TIMESTAMP_COLUMN]

print("=== KIỂM TRA THỜI GIAN ===")
print(f"  Min        : {ts.min()}")
print(f"  Max        : {ts.max()}")
print(f"  Span       : {(ts.max() - ts.min()).days} ngày")
print(f"  Tăng nghiêm ngặt: {ts.is_monotonic_increasing}")
print(f"  NaT        : {int(ts.isna().sum())}")
print(f"  Timezone   : {ts.dt.tz}")

# Gời trùng trục thời gian là lỗi ingestion, không phải hiện tượng tự nhiên.
if not ts.is_monotonic_increasing:
    n_bad = int((ts.diff() <= pd.Timedelta(0)).sum())
    print(f"⚠️  Trục thời gian KHÔNG tăng nghiêm ngặt — {n_bad} vị trí không tăng.")

# Mất giờ trong lưới 1h là dấu hiệu ingestion rớt hàng, cần nêu ra chứ không im lặng.
# So sánh qua `.dt.total_seconds()` thay vì `!= pd.Timedelta(...)`: so sánh trực
# tiếp với Series timedelta kích hoạt DeprecationWarning của pandas/numpy.
gaps = ts.diff().dt.total_seconds().dropna()
gap_hours = gaps[gaps != 3600.0]
if len(gap_hours):
    worst = pd.to_timedelta(gap_hours.max(), unit="h")
    print(f"⚠️  {len(gap_hours)} khoảng trống ≠ 1h (lớn nhất: {worst})")
else:
    print("✓ Lưới thời gian liên tục 1h, không có khoảng trống")

# `strftime` thay vì `.dt.to_period()`: to_period() bỏ mất timezone và phát
# UserWarning. Cách này giữ nguyên nhãn tháng theo giờ UTC+7.
print("\nSố quan sát theo tháng:")
display(ts.dt.strftime("%Y-%m").value_counts().sort_index())

In [ ]:
print("=== KIỂM TRA STATION ===")
print(f"  Cột       : {STATION_COLUMN}")
print(f"  Số trạm   : {df[STATION_COLUMN].nunique()}")
display(df[STATION_COLUMN].value_counts())

## 5. Kiểm tra kết quả cleaning

Mọi ngưỡng dưới đây **import từ** `src/cleaning.py` / `src/data_collection.py`. Notebook này không
định nghĩa lại luật — nếu `src/cleaning.py` đổi ngưỡng, các con số ở đây đổi theo.

**NaN là kết quả hợp lệ của cleaning, không phải lỗi.** Các kiểm tra dưới đây vì vậy chỉ báo cáo
số lượng, không khẳng định "phải bằng 0".

In [ ]:
print("=== KIỂM TRA KẾT QUẢ CLEANING ===\n")

# --- 1. Mã lỗi số ẩn (disguised missing) ----------------------------------
# `normalize_disguised_missing()` trong src/cleaning.py chuyển các mã này thành NaN.
# Sau cleaning, chúng phải bằng 0 — nếu còn, nghĩa là cột chưa đi qua cleaning.
print(f"1. Mã lỗi {DISGUISED_NUMERIC_CODES} (phải = 0 sau cleaning):")
for code_val in DISGUISED_NUMERIC_CODES:
    n_cells = int((df == code_val).sum().sum())
    status = "✓" if n_cells == 0 else "❌ còn sót"
    print(f"   {code_val}: {n_cells} cells  {status}")

# --- 2. Ràng buộc vật lý của kênh khí tượng --------------------------------
# Cùng bộ ràng buộc mà `src/data_collection.py` dùng khi ingest. Giá trị âm ở
# kênh nào đó là vi phạc, KHÔNG phải dữ liệu hợp lệ.
print("\n2. Ràng buộc vật lý (nguồn: WEATHER_PHYSICAL_BOUNDS):")
for col, bounds in WEATHER_PHYSICAL_BOUNDS.items():
    if col not in df.columns:
        print(f"   {col:<18} — không có trong artifact")
        continue
    lo, hi = bounds["min"], bounds["max"]
    series = df[col]
    n_lo = int((series < lo).sum())
    n_hi = int((series > hi).sum())
    unit = bounds.get("unit", "")
    status = "✓" if n_lo == n_hi == 0 else f"❌ {n_lo} dưới, {n_hi} trên"
    print(f"   {col:<18} [{lo}, {hi}] {unit:<8} "
          f"min={series.min():.2f} max={series.max():.2f}  {status}")

In [ ]:
# --- 3. PM2.5 ⊆ PM10 -------------------------------------------------------
# `enforce_pm_subset_constraint()` trong src/cleaning.py áp dụng
#     PM2.5 ≤ PM10 + ε     với ε = PM_AERODYNAMIC_EPSILON_UG_M3
# Dùng ε, KHÔNG dùng so sánh nghiêm ngặt pm25 > pm10: sai số đo thiết bị là hợp
# lệ, nên so sánh không có dung sai sẽ báo động giả trên dữ liệu ĐÃ sạch.
if TARGET_CANDIDATE in df.columns and "pm10" in df.columns:
    pair = df[[TARGET_CANDIDATE, "pm10"]].dropna()
    strict = int((pair[TARGET_CANDIDATE] > pair["pm10"]).sum())
    with_eps = int((pair[TARGET_CANDIDATE] > pair["pm10"] + PM_AERODYNAMIC_EPSILON_UG_M3).sum())
    print(f"3. PM2.5 ⊆ PM10 (ε = {PM_AERODYNAMIC_EPSILON_UG_M3} µg/m³):")
    print(f"   Cặp quan sát hợp lệ     : {len(pair):,}")
    print(f"   Nghiêm ngặt pm25 > pm10  : {strict}")
    print(f"   Có xét ε (đúng luật)     : {with_eps}  "
          f"{'✓ khớp luật của src/cleaning.py' if with_eps == 0 else '❌ còn vi phạm'}")
    if strict > with_eps:
        print(f"   → {strict - with_eps} cặp lệch trong dung sai đo thiết bị, "
              f"KHÔNG phải nghịch đảo khí động học.")
else:
    print(f"3. Bỏ qua: cần cả `{TARGET_CANDIDATE}` và `pm10`")

In [ ]:
# --- 4. Trùng khóa quan sát -------------------------------------------------
# Khoá canonical = (station_id, timestamp) cho tập đa trạm, chỉ timestamp cho
# tập đơn trạm (docs/data_dictionary.md §3.3). Kiểm tra chỉ theo timestamp sẽ
# báo trùng giả ngay khi có trạm thứ hai.
key = [STATION_COLUMN] if df[STATION_COLUMN].nunique() > 1 else [TIMESTAMP_COLUMN]
n_dup = int(df.duplicated(subset=key).sum())
print(f"4. Trùng khoá {key}:")
print(f"   Số bản ghi trùng: {n_dup}  {'✓' if n_dup == 0 else '❌'}")

n_stations = df[STATION_COLUMN].nunique()
if n_stations == 1:
    print("   (tập đơn trạm — khoá chỉ cần timestamp theo §3.3)")
else:
    print(f"   (tập đa trạm — {n_stations} trạm — khoá phải gồm cả station_id)")

In [ ]:
# --- 5. Cờ chẩn đoán --------------------------------------------------------
# Cờ chẩn đoán KHÔNG phải imputation. Ở đây chỉ báo cáo tỉ lệ — quyết định có
# impute được hay không thuộc Issue #7, không thuộc notebook xem dữ liệu.
print("5. Cờ chẩn đoán (không phải giá trị đã điền):")
for flag, desc in [
    (PM25_MISSING_FLAG, "trạm ngừng phát (báo cáo gap)"),
    (STUCK_SENSOR_FLAG, "cảm biến bị kẹt"),
    (HIGH_HUMIDITY_FLAG, f"sương mù quang học (RH > {HIGH_HUMIDITY_FOG_THRESHOLD_PCT}%)"),
]:
    if flag in df.columns:
        n = int(df[flag].sum())
        print(f"   {flag:<22} {n:>5} giờ  ({100 * n / len(df):.2f}%)  — {desc}")
    else:
        print(f"   {flag:<22} không có trong artifact")

# NaN ở cột target là hệ quả của các cờ trên, không phải hụt dữ liệu ngẫu nhiên.
if TARGET_CANDIDATE in df.columns:
    n_target_nan = int(df[TARGET_CANDIDATE].isna().sum())
    print(f"\n   {TARGET_CANDIDATE} = NaN: {n_target_nan} "
          f"({100 * n_target_nan / len(df):.2f}%) — kết quả của cleaning, không phải lỗi")

## 6. Feature được tạo thêm

Danh sách lấy từ `src/cleaning_pipeline.py` — không liệt kê tay.

In [ ]:
engineered = {
    **{c: "cyclical encoding — chu kỳ ngày/tháng" for c in CYCLICAL_FEATURES},
    **{c: f"cờ chẩn đoán (xem mục 5)" for c in DIAGNOSTIC_FEATURES + TARGET_DERIVED_FLAGS},
}
print("=== FEATURE ĐƯỢC TẠO THÊM ===")
present = []
for col, desc in engineered.items():
    if col in df.columns:
        present.append(col)
        print(f"✓ {col:<22} {desc}")
        print(f"    min={df[col].min():.4f}  max={df[col].max():.4f}  mean={df[col].mean():.4f}")
    else:
        print(f"✗ {col:<22} KHÔNG có trong artifact")
print(f"\n{len(present)}/{len(engineered)} feature kỳ vọng hiện diện")

## 7. Train/Test split

Artifact parquet **không mang theo** thông tin mốc cắt: `03_transformation_pipeline.ipynb` gọi
`freeze_dataset_with_audit(..., additional_info={"split_timestamp": ...})` nhưng chỉ in ra console,
không ghi sidecar. Vì vậy notebook này **không bịa** con số — nó đọc sidecar nếu có, và nếu không thì
nói thẳng là không có.

Quyết định mốc cắt được ghi ở `docs/roadmap.md` → *"Quyết định 1 — Điểm cắt Train/Test"*.

In [ ]:
print("=== TRAIN/TEST SPLIT ===")

# Sidecar KHÔNG thuộc tầng raw: `data/raw/metadata.json` là hồ sơ nguồn bất biến
# (Three-tier Raw Data Policy) và không chứa quyết định split của tầng processed.
SIDECAR = OUTPUT_PATH.with_suffix(".metadata.json")

if SIDECAR.exists():
    split_meta = json.loads(SIDECAR.read_text(encoding="utf-8"))
    print(f"✓ Đọc split từ sidecar: {SIDECAR.relative_to(REPO_ROOT)}")
    print(f"  split_timestamp: {split_meta.get('split_timestamp', '—')}")
    split_ts = pd.Timestamp(split_meta["split_timestamp"]) if "split_timestamp" in split_meta else None
else:
    split_ts = None
    print("ℹ️  Artifact KHÔNG mang thông tin split.")
    print(f"   Không có sidecar: {SIDECAR.relative_to(REPO_ROOT)}")
    print("   Mốc cắt đã chốt tại docs/roadmap.md → 'Quyết định 1 — Điểm cắt Train/Test'.")
    print("   (03_transformation_pipeline.ipynb chỉ in ra console, chưa ghi sidecar.)")

# Những gì verify được TRỰC TIẾP từ artifact, không cần sidecar.
ts = df[TIMESTAMP_COLUMN]
print(f"\nTổng quan (đọc trực tiếp từ artifact):")
print(f"  rows     : {len(df):,}")
print(f"  span     : {ts.min()} → {ts.max()}")
print(f"  stations : {df[STATION_COLUMN].nunique()}")

if split_ts is not None:
    if ts.dt.tz is not None and split_ts.tz is None:
        split_ts = split_ts.tz_localize(ts.dt.tz)
    n_train = int((ts < split_ts).sum())
    n_test = int((ts >= split_ts).sum())
    print(f"\n  split_timestamp: {split_ts}")
    print(f"  train: {n_train:,} ({100 * n_train / len(df):.1f}%)")
    print(f"  test : {n_test:,} ({100 * n_test / len(df):.1f}%)")

## 8. Trực quan hoá kiểm tra dữ liệu

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts, df[TARGET_CANDIDATE], linewidth=0.8, alpha=0.8)
ax.set_title("PM2.5 theo thời gian", fontsize=12, fontweight="bold")
ax.set_xlabel("Thời gian (Asia/Ho_Chi_Minh, UTC+7)")
ax.set_ylabel("PM2.5 (µg/m³)")
ax.margins(x=0.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df[TARGET_CANDIDATE].dropna().hist(bins=50, ax=ax, edgecolor="black", alpha=0.7)
ax.set_title("Phân phối PM2.5", fontsize=12, fontweight="bold")
ax.set_xlabel("PM2.5 (µg/m³)")
ax.set_ylabel("Số giờ")
plt.tight_layout()
plt.show()

print(f"skew    = {df[TARGET_CANDIDATE].skew():.4f}")
print(f"kurtosis= {df[TARGET_CANDIDATE].kurtosis():.4f}")

In [ ]:
check_cols = [TARGET_CANDIDATE, "pm10"] + [c for c in NUMERIC_FEATURES if c in df.columns]
missing_counts = df[check_cols].isna().sum().sort_values()
if int(missing_counts.sum()) == 0:
    print("✓ Không có NaN ở bất kỳ cột quan trọng nào.")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    missing_counts.plot(kind="barh", ax=ax, color="coral")
    ax.set_title("Số giờ bị cleaning để thành NaN, theo cột", fontsize=12, fontweight="bold")
    ax.set_xlabel("Số giờ NaN")
    # `kind="barh"` đặt nhãn trục y theo VỊ TRÍ, không phải theo tên cột — nạp
    # số vào `ax.text` phải dùng chỉ số, dùng tên cột sẽ ném ConversionError.
    for pos, (col, n) in enumerate(missing_counts.items()):
        if n > 0:
            ax.text(n, pos, f" {n}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()
    print("NaN ở đây là kết quả của cleaning, không phải hụt dữ liệu.")

In [ ]:
# PM2.5 theo giờ trong ngày — kiểm tra chu kỳ ngày có thật không.
# Dùng giờ local, không dùng giờ UTC: dữ liệu đã chuẩn hoá về Asia/Ho_Chi_Minh.
hourly = df.dropna(subset=[TARGET_CANDIDATE]).groupby(ts.dt.hour)[TARGET_CANDIDATE]
profile = pd.DataFrame({
    "mean_pm25": hourly.mean().round(1),
    "n": hourly.size(),
})
display(profile)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(profile.index, profile["mean_pm25"], marker="o", linewidth=2)
ax.set_title("PM2.5 trung bình theo giờ trong ngày (giờ UTC+7)", fontsize=12, fontweight="bold")
ax.set_xlabel("Giờ trong ngày")
ax.set_ylabel("PM2.5 trung bình (µg/m³)")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

## 9. Final preview

Luôn giữ `timestamp`, `station_id` và target trong phần hiển thị — cắt cột theo thứ tự rồi lấy
`df.columns[:15]` có thể làm mất đúng những cột quan trọng nhất.

In [ ]:
must_show = [c for c in (TIMESTAMP_COLUMN, STATION_COLUMN, TARGET_CANDIDATE) if c in df.columns]
rest = [c for c in df.columns if c not in must_show]

budget = 15
if len(df.columns) > budget:
    display_cols = must_show + rest[: max(0, budget - len(must_show))]
    print(f"Dataset có {len(df.columns)} cột — hiển thị {len(display_cols)} cột ưu tiên.")
    print(f"  Giữ bắt buộc : {must_show}")
    print(f"  Bị ẩn        : {rest[max(0, budget - len(must_show)):]}  (dùng df.head() để xem đầy đủ)")
else:
    display_cols = list(df.columns)

display(df.head(20)[display_cols])

---
**Kết luận:** artifact này là output của `merge → freeze → split → fit` ở Issue #7. NaN trong
`pm25` là quan sát bị loại có chủ đích, không phải hụt dữ liệu. Ba cờ chẩn đoán cho biết vì sao.
Mọi ngưỡng ở mục 5 đến từ `src/cleaning.py` — nếu số ở đây mâu thuẫn với `docs/cleaning_log.md`,
hãy tin `src/cleaning.py` và báo lại để sửa notebook này.